# 09 — Hybrid test-set evaluation

Mirrors notebook 07 (held-out test eval of the 108 primary adapters), but for
the 36 `hybrid_ia3_lora` runs from notebook 08. Loads each saved LoRA adapter
+ classifier via `PeftModel.from_pretrained`, then re-attaches IA3 hooks and
restores `ia3_vectors.pt` on top — same `attach_ia3_vectors` as 08, so the
attached module set and shapes line up exactly with what was trained.

In [ ]:
# ------------------------------
# 1. Environment Setup
# ------------------------------
!pip uninstall -y torchao -q
!pip install -q peft --no-deps
!pip install -q accelerate

import torch, torch.nn as nn, os, gc, pandas as pd, numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PeftModel
from datasets import Dataset
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

In [ ]:
# ------------------------------
# 2. Configuration & paths
# ------------------------------
MODEL_NAME = "xlm-roberta-base"
NUM_LABELS = 3
MAX_LENGTH = 128
BATCH_SIZE = 64
METHOD_NAME = "hybrid_ia3_lora"

DATA_ROOT = "/kaggle/input/notebooks/venkatkolluu/02-data-preprocessingv2/data/processed"
# Once 08 is committed as a Kaggle notebook output/dataset, its adapters + csv live here.
# Adjust this path to wherever your committed 08 output actually lands.
PRIMARY_RESULTS_ROOT = "/kaggle/input/notebooks/venkatkolluu/08-hybrid-formal"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

sweep_df = pd.read_csv(f"{PRIMARY_RESULTS_ROOT}/hybrid_experiment_results.csv")
print(f"Loaded {len(sweep_df)} hybrid runs from {PRIMARY_RESULTS_ROOT}/hybrid_experiment_results.csv")

In [ ]:
# ------------------------------
# 3. Held-out test data
# ------------------------------
def build_test_loader(language):
    test_df = pd.read_parquet(f"{DATA_ROOT}/{language}/test.parquet")
    def tokenize(batch):
        return tokenizer(batch["premise"], batch["hypothesis"], truncation=True,
                          padding="max_length", max_length=MAX_LENGTH)
    ds = Dataset.from_pandas(test_df).rename_column("label", "labels").map(tokenize, batched=True)
    keep = ["input_ids", "attention_mask", "labels"]
    ds = ds.remove_columns([c for c in ds.column_names if c not in keep])
    ds.set_format("torch")
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False)

loaders = {"hi": build_test_loader("hi"), "te": build_test_loader("te")}
print(f"Test sets — Hindi: {len(loaders['hi'].dataset)}, Telugu: {len(loaders['te'].dataset)}")

In [ ]:
# ------------------------------
# 4. IA3 hooks — identical to notebook 08, needed to reconstruct the hybrid model
# ------------------------------
def attach_ia3_vectors(model, hidden_size, intermediate_size):
    vectors = {}
    handles = []

    def make_output_hook(vec):
        def hook(module, inputs, output):
            return output * vec
        return hook

    def make_input_hook(vec):
        def hook(module, inputs):
            return (inputs[0] * vec,)
        return hook

    for name, module in model.named_modules():
        if name.endswith("attention.self.key"):
            vec = nn.Parameter(torch.ones(hidden_size))
            vectors[name] = vec
            handles.append(module.register_forward_hook(make_output_hook(vec)))
        elif name.endswith("intermediate.dense"):
            vec = nn.Parameter(torch.ones(hidden_size))
            vectors[name] = vec
            handles.append(module.register_forward_pre_hook(make_input_hook(vec)))
        elif name.endswith("output.dense") and "attention" not in name:
            vec = nn.Parameter(torch.ones(intermediate_size))
            vectors[name] = vec
            handles.append(module.register_forward_pre_hook(make_input_hook(vec)))

    safe_dict = nn.ParameterDict({k.replace(".", "__"): v for k, v in vectors.items()})
    model.ia3_vectors = safe_dict
    return handles

def load_hybrid_model(adapter_path):
    base = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS).cuda()
    model = PeftModel.from_pretrained(base, adapter_path)  # restores LoRA + classifier

    handles = attach_ia3_vectors(model, base.config.hidden_size, base.config.intermediate_size)
    ia3_path = os.path.join(adapter_path, "ia3_vectors.pt")
    ia3_state = torch.load(ia3_path, map_location="cuda", weights_only=True)
    model.ia3_vectors.load_state_dict(ia3_state)

    model = model.cuda()
    model.eval()
    return model, handles

In [ ]:
# ------------------------------
# 5. Evaluation loop (resumable)
# ------------------------------
results_file = "/kaggle/working/hybrid_test_results.csv"

completed_keys = set()
if os.path.exists(results_file):
    existing_df = pd.read_csv(results_file)
    for _, row in existing_df.iterrows():
        completed_keys.add((row["method"], row["language"], int(row["budget"]), int(row["seed"])))
    print(f"Found {len(completed_keys)} previously completed evaluations. Resuming...")

total_runs = len(sweep_df)

for idx, row in sweep_df.iterrows():
    method = str(row["method"]); lang = str(row["language"])
    budget = int(row["budget"]); seed = int(row["seed"])

    key = (method, lang, budget, seed)
    if key in completed_keys:
        print(f"[{idx+1}/{total_runs}] SKIP (done): {method} | {lang} | budget={budget} | seed={seed}")
        continue

    adapter_path = f"{PRIMARY_RESULTS_ROOT}/adapters/{method}/{lang}/budget{budget}_seed{seed}"
    print(f"[{idx+1}/{total_runs}] Evaluating {method} | {lang} | budget={budget} | seed={seed}")

    try:
        model, handles = load_hybrid_model(adapter_path)
        preds, labels = [], []
        with torch.no_grad():
            for batch in loaders[lang]:
                batch = {k: v.cuda() for k, v in batch.items()}
                outputs = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
                preds.extend(torch.argmax(outputs.logits, dim=1).cpu().numpy())
                labels.extend(batch["labels"].cpu().numpy())

        acc = accuracy_score(labels, preds)
        f1 = f1_score(labels, preds, average="macro")
        res = {"method": method, "language": lang, "budget": budget, "seed": seed,
               "test_accuracy": round(acc, 6), "test_macro_f1": round(f1, 6)}
        pd.DataFrame([res]).to_csv(results_file, mode="a", header=not os.path.exists(results_file), index=False)
        print(f"   -> Test Acc: {acc:.4f} | Test Macro-F1: {f1:.4f}")
    except Exception as e:
        print(f"   -> ERROR: {e}")
    finally:
        if "model" in locals():
            for h in handles: h.remove()
            del model
        gc.collect(); torch.cuda.empty_cache()

print("\nHybrid test-set evaluation complete!")
if os.path.exists(results_file):
    final_df = pd.read_csv(results_file)
    print(f"Total evaluated runs saved: {len(final_df)}/{total_runs}")